YOLOv8n study:

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data="/content/drive/MyDrive/road_surface_project/data/road_damage_dataset_3000/data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    seed=42,

    #Аугментация
    fliplr=0.5,
    mosaic=1.0,
    scale=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    project="road_damage_project",
    name="yolov8n_baseline"
)

YOLO11n study:

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

results = model.train(
    data="/content/drive/MyDrive/road_surface_project/data/road_damage_dataset_3000/data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    seed=42,

    #Аугментация
    fliplr=0.5,
    mosaic=1.0,
    scale=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    project="road_damage_project",
    name="yolo11n_baseline"
)

YOLO26n study:

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")

results = model.train(
    data="/content/drive/MyDrive/road_surface_project/data/road_damage_dataset_3000/data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    seed=42,

    #Аугментация
    fliplr=0.5,
    mosaic=1.0,
    scale=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    project="road_damage_project",
    name="yolo26n_baseline"
)

SSD study:

In [ ]:
#Обучение модели SSD

# import + setup
import torch
import torchvision
import os
import cv2
import yaml
import numpy as np
import random
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# воспроизводимость
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

# path
DATASET_PATH = "/content/drive/MyDrive/road_surface_project/data/road_damage_dataset_3000"
SAVE_PATH = "/content/drive/MyDrive/road_surface_project/runs/exp4_SSD"

os.makedirs(SAVE_PATH, exist_ok=True)

print("Saving to:", SAVE_PATH)

#load yaml
with open(f"{DATASET_PATH}/data.yaml", "r") as f:
    data = yaml.safe_load(f)

NUM_CLASSES = len(data["names"]) + 1
print("Classes:", NUM_CLASSES)

# датасет из YOLO в SSD формат
class YOLODataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, label_dir):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.images = [f for f in os.listdir(img_dir) if f.endswith(".jpg") or f.endswith(".png")]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        img = cv2.resize(img, (300, 300))
        h, w, _ = img.shape

        label_path = os.path.join(
            self.label_dir,
            img_name.replace(".jpg", ".txt").replace(".png", ".txt")
        )

        boxes = []
        labels = []

        if os.path.exists(label_path) and os.path.getsize(label_path) > 0:
            with open(label_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue

                    cls, x, y, bw, bh = map(float, parts)

                    x1 = (x - bw/2) * w
                    y1 = (y - bh/2) * h
                    x2 = (x + bw/2) * w
                    y2 = (y + bh/2) * h

                    boxes.append([x1, y1, x2, y2])
                    labels.append(int(cls) + 1)

        if len(boxes) == 0:
          boxes = torch.zeros((0, 4), dtype=torch.float32)
          labels = torch.zeros((0,), dtype=torch.int64)

        img = torch.from_numpy(img).float().permute(2, 0, 1) / 255.0

        if len(boxes) > 0:
          boxes = torch.tensor(boxes, dtype=torch.float32)
          labels = torch.tensor(labels, dtype=torch.int64)
        else:
          boxes = torch.zeros((0, 4), dtype=torch.float32)
          labels = torch.zeros((0,), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx])
        }

        return img, target

# data loaders
train_dataset = YOLODataset(
    f"{DATASET_PATH}/train/images",
    f"{DATASET_PATH}/train/labels"
)

val_dataset = YOLODataset(
    f"{DATASET_PATH}/valid/images",
    f"{DATASET_PATH}/valid/labels"
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=lambda x: tuple(zip(*x)),
    num_workers=2,
    pin_memory=True
)

val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=lambda x: tuple(zip(*x)),
    num_workers=2,
    pin_memory=True
)
# модель SSD
model = torchvision.models.detection.ssd300_vgg16(weights="DEFAULT")

from torchvision.models.detection.ssd import SSDClassificationHead

in_channels = [512, 1024, 512, 256, 256, 256]
num_anchors = model.anchor_generator.num_anchors_per_location()

model.head.classification_head = SSDClassificationHead(
    in_channels=in_channels,
    num_anchors=num_anchors,
    num_classes=NUM_CLASSES
)

model = model.to(device)

# оптимизация
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.001,
    momentum=0.9,
    weight_decay=0.0005
)

# training
NUM_EPOCHS = 15
loss_history = []
best_loss = float("inf")

scaler = torch.cuda.amp.GradScaler()

for epoch in range(NUM_EPOCHS):

    model.train()
    epoch_loss = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")

    for imgs, targets in loop:

        imgs = [img.to(device) for img in imgs]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast():
            loss_dict = model(imgs, targets)
            loss = sum(loss_dict.values())

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    avg_loss = epoch_loss / len(train_loader)
    loss_history.append(avg_loss)

    print(f"Epoch {epoch+1} Loss: {avg_loss:.4f}")

    torch.save(model.state_dict(), f"{SAVE_PATH}/epoch_{epoch+1}.pth")

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), f"{SAVE_PATH}/best_model.pth")

#график лоссов
plt.plot(loss_history)
plt.title("SSD Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid()

plt.savefig(f"{SAVE_PATH}/loss_curve.png")
plt.show()

Faster R-CNN study:

In [ ]:
#Обучение модели Faster R-CNN

# ======================================
# 1. IMPORTS
# ======================================
import torch
import torchvision
import random
import numpy as np
import os
import cv2
import yaml
import matplotlib.pyplot as plt
from tqdm import tqdm
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# ======================================
# 2. SPEED + REPRODUCIBILITY
# ======================================
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

torch.backends.cudnn.deterministic = False   # faster
torch.backends.cudnn.benchmark = True        # faster on T4

# ======================================
# 3. DEVICE
# ======================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ======================================
# 4. PATHS
# ======================================
DATASET_PATH = "/content/drive/MyDrive/road_surface_project/data/road_damage_dataset_3000"
SAVE_PATH = "/content/drive/MyDrive/road_surface_project/results/faster_rcnn"

os.makedirs(SAVE_PATH, exist_ok=True)

# ======================================
# 5. YAML
# ======================================
with open(f"{DATASET_PATH}/data.yaml", "r") as f:
    data = yaml.safe_load(f)

NUM_CLASSES = len(data["names"]) + 1

# ======================================
# 6. DATASET (FAST YOLO PARSER)
# ======================================
class YOLODataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, label_dir):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.images = [
            f for f in os.listdir(img_dir)
            if f.endswith(".jpg") or f.endswith(".png")
        ]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # 🔥 SPEED BOOST: resize
        img = cv2.resize(img, (512, 512))

        h, w, _ = img.shape

        label_path = os.path.join(
            self.label_dir,
            img_name.replace(".jpg", ".txt").replace(".png", ".txt")
        )

        boxes = []
        labels = []

        if os.path.exists(label_path) and os.path.getsize(label_path) > 0:
            with open(label_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue

                    cls, x, y, bw, bh = map(float, parts)

                    x1 = (x - bw / 2) * w
                    y1 = (y - bh / 2) * h
                    x2 = (x + bw / 2) * w
                    y2 = (y + bh / 2) * h

                    boxes.append([x1, y1, x2, y2])
                    labels.append(int(cls) + 1)

        # skip empty
        if len(boxes) == 0:
            return self.__getitem__((idx + 1) % len(self.images))

        # 🔥 FAST tensor conversion
        img = torch.from_numpy(img).float().permute(2, 0, 1) / 255.0

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx])
        }

        return img, target

# ======================================
# 7. DATA LOADERS (FAST)
# ======================================
train_dataset = YOLODataset(
    f"{DATASET_PATH}/train/images",
    f"{DATASET_PATH}/train/labels"
)

val_dataset = YOLODataset(
    f"{DATASET_PATH}/valid/images",
    f"{DATASET_PATH}/valid/labels"
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=lambda x: tuple(zip(*x)),
    num_workers=2,
    pin_memory=True
)

val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=lambda x: tuple(zip(*x)),
    num_workers=2,
    pin_memory=True
)

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))

# ======================================
# 8. MODEL
# ======================================
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")

in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)

model = model.to(device)

# ======================================
# 9. OPTIMIZER
# ======================================
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.005,
    momentum=0.9,
    weight_decay=0.0005
)

lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=3,
    gamma=0.1
)

# ======================================
# 10. AMP (🔥 SPEED BOOST)
# ======================================
scaler = torch.cuda.amp.GradScaler()

# ======================================
# 11. TRAINING
# ======================================
NUM_EPOCHS = 15   # faster version
loss_history = []
best_loss = float("inf")

for epoch in range(NUM_EPOCHS):

    model.train()
    epoch_loss = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")

    for imgs, targets in loop:

        imgs = [img.to(device) for img in imgs]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        optimizer.zero_grad()

        # 🔥 AMP forward
        with torch.cuda.amp.autocast():
            loss_dict = model(imgs, targets)
            loss = sum(loss for loss in loss_dict.values())

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    lr_scheduler.step()

    avg_loss = epoch_loss / len(train_loader)
    loss_history.append(avg_loss)

    print(f"\nEpoch {epoch+1}: Loss = {avg_loss:.4f}")

    # save best model
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), f"{SAVE_PATH}/best_model.pth")
        print("🔥 Best model saved!")

# ======================================
# 12. LOSS PLOT
# ======================================
plt.plot(loss_history)
plt.title("Faster R-CNN Loss (FAST VERSION)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid()
plt.savefig(f"{SAVE_PATH}/loss_curve.png")
plt.show()

print("Training finished!")
print("Saved to:", SAVE_PATH)

DETR study:

In [ ]:
#Обучение модели DETR

# ======================================
# 1. IMPORTS
# ======================================
import torch
import os
import cv2
import yaml
import random
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import DetrImageProcessor, DetrForObjectDetection

# ======================================
# 2. DEVICE
# ======================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ======================================
# 3. SEED
# ======================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

torch.backends.cudnn.benchmark = True

# ======================================
# 4. PATHS
# ======================================
DATASET_PATH = "/content/drive/MyDrive/road_surface_project/data/road_damage_dataset_3000"
SAVE_PATH = "/content/drive/MyDrive/road_surface_project/runs/exp5_DETR"

os.makedirs(SAVE_PATH, exist_ok=True)

print("Saving to:", SAVE_PATH)

# ======================================
# 5. DATA
# ======================================
with open(f"{DATASET_PATH}/data.yaml", "r") as f:
    data = yaml.safe_load(f)

NUM_CLASSES = len(data["names"])
print("Classes:", NUM_CLASSES)

processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")

# ======================================
# 6. DATASET (FIXED COCO FORMAT)
# ======================================
class YOLODataset(Dataset):
    def __init__(self, img_dir, label_dir):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.images = [
            f for f in os.listdir(img_dir)
            if f.endswith(".jpg") or f.endswith(".png")
        ]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # FIX SIZE
        image = cv2.resize(image, (800, 800))
        h, w = 800, 800

        label_path = os.path.join(
            self.label_dir,
            img_name.replace(".jpg", ".txt").replace(".png", ".txt")
        )

        annotations = []

        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                for line in f:
                    cls, x, y, bw, bh = map(float, line.split())

                    x1 = (x - bw/2) * w
                    y1 = (y - bh/2) * h
                    x2 = (x + bw/2) * w
                    y2 = (y + bh/2) * h

                    bbox_w = x2 - x1
                    bbox_h = y2 - y1

                    area = bbox_w * bbox_h  # 🔥 FIX

                    annotations.append({
                        "bbox": [x1, y1, bbox_w, bbox_h],
                        "category_id": int(cls),
                        "area": float(area),
                        "iscrowd": 0
                    })

        encoding = processor(
            images=image,
            annotations={
                "image_id": idx,
                "annotations": annotations
            },
            return_tensors="pt"
        )

        return {
            "pixel_values": encoding["pixel_values"].squeeze(0),
            "labels": encoding["labels"][0]
        }

# ======================================
# 7. COLLATE FN
# ======================================
def collate_fn(batch):
    pixel_values = [b["pixel_values"] for b in batch]
    labels = [b["labels"] for b in batch]

    return {
        "pixel_values": torch.stack(pixel_values),
        "labels": labels
    }
    # ======================================
# 8. DATALOADER
# ======================================
train_dataset = YOLODataset(
    f"{DATASET_PATH}/train/images",
    f"{DATASET_PATH}/train/labels"
)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    collate_fn=collate_fn
)

# ======================================
# 9. MODEL
# ======================================
model = DetrForObjectDetection.from_pretrained(
    "facebook/detr-resnet-50",
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True
)

model.to(device)

# ======================================
# 10. OPTIMIZER
# ======================================
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

# ======================================
# 11. TRAIN LOOP
# ======================================
EPOCHS = 30
loss_history = []

for epoch in range(EPOCHS):

    model.train()
    epoch_loss = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for batch in loop:

        pixel_values = batch["pixel_values"].to(device)
        labels = [{k: v.to(device) for k, v in t.items()} for t in batch["labels"]]

        outputs = model(
            pixel_values=pixel_values,
            labels=labels
        )

        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    avg_loss = epoch_loss / len(train_loader)
    loss_history.append(avg_loss)

    print(f"Epoch {epoch+1} Loss: {avg_loss:.4f}")

    torch.save(model.state_dict(), f"{SAVE_PATH}/epoch_{epoch+1}.pth")

# ======================================
# 12. SAVE MODEL
# ======================================
torch.save(model.state_dict(), f"{SAVE_PATH}/best_model.pth")

# ======================================
# 13. LOSS PLOT
# ======================================
plt.plot(loss_history)
plt.title("DETR Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid()
plt.savefig(f"{SAVE_PATH}/loss_curve.png")
plt.show()